# Barbershop • ทดลองเปลี่ยนทรงผมและสีผมบน Google Colab
คู่มือภาษาไทย | อ้างอิงงาน Zhu et al., 2021 และโค้ดทางการ ZPdesu/Barbershop

**เริ่มต้น:** อัปโหลดไฟล์นี้เข้า https://colab.research.google.com → Runtime → Change runtime type → เลือก GPU (เช่น T4) → รันจากบนลงล่าง ทีละเซลล์ โดยรอแต่ละเซลล์เสร็จ

ใช้ภาพ 3 รูป: **face** = คนที่ต้องการเปลี่ยนผม, **shape** = ทรงผมที่ต้องการ, **color** = ภาพบุคคลที่มีสี/ลักษณะเส้นผมที่ต้องการ (ไม่ใช่แถบสี)

ผลลัพธ์เป็นภาพใบหน้าที่จัดแนวแล้วขนาด 1024 × 1024 ไม่ใช่ภาพเต็มตัวเดิม โมเดลอาจเปลี่ยนรายละเอียดใบหน้าหรือพื้นหลังบางส่วน ควรใช้ภาพคนเดียว หน้าชัด และมุมใบหน้าใกล้กัน

**สถานะการตรวจสอบ:** ตรวจ API กับ source code และตรวจ syntax ของ notebook แล้ว แต่ยังไม่ได้รันสร้างภาพบน Colab GPU จริงในสภาพแวดล้อมผู้จัดทำ จึงยังยืนยันการดาวน์โหลดน้ำหนักและความเข้ากันได้ของ runtime ทุกเวอร์ชันไม่ได้

## 1. ตรวจ GPU และติดตั้ง
เก็บ PyTorch/torchvision คู่ที่ Colab เตรียมไว้ เพื่อลดปัญหา CUDA ไม่ตรงกัน ติดตั้งเฉพาะแพ็กเกจเพิ่ม หากเพิ่งเปลี่ยน runtime ให้เริ่มรันใหม่ตั้งแต่เซลล์นี้

In [ ]:
import sys, subprocess, os
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'กรุณาเลือก Runtime > Change runtime type > GPU ก่อน'
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown==5.2.0',
                'ninja', 'scipy', 'scikit-image', 'scikit-learn', 'tqdm', 'requests', 'matplotlib'], check=True)
try:
    import dlib
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'dlib==19.24.6'], check=True)
import torchvision
print('Torchvision:', torchvision.__version__)
ROOT = Path('/content/Barbershop')

## 2. โหลดโค้ดและปรับความเข้ากันได้
ล็อก commit ของโค้ดที่ตรวจอ่านไว้ ใช้การคำนวณ StyleGAN2 แบบ PyTorch native แทน CUDA extension รุ่นเก่า เพื่อไม่ต้อง compile kernel เก่า (อาจช้ากว่า kernel ที่ compile สำเร็จ) ไม่เปลี่ยนสถาปัตยกรรมหรือฝึกโมเดลใหม่

ปรับ Pillow API, เงื่อนไข mask เป็น boolean, ค่า loss ให้เป็น scalar, ป้องกันหาร mask ว่าง และปล่อย GPU memory ระหว่างช่วงงาน บันทึกรายการแก้ไว้ใน `colab_compatibility.json`

In [ ]:
import ast, json, textwrap
COMMIT = '088d7d0f0623b348732fc15d8fbdea8b8cb730c7'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ZPdesu/Barbershop.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', COMMIT], check=True)
os.chdir(ROOT)
# Restore only files patched by this notebook; preserve inputs and results.
patch_paths = ['utils/shape_predictor.py', 'models/Alignment.py', 'models/Blending.py',
               'losses/masked_lpips/networks_basic.py', 'models/stylegan2/op/upfirdn2d.py',
               'models/stylegan2/op/fused_act.py', 'main.py']
subprocess.run(['git', 'restore', '--source='+COMMIT, '--', *patch_paths], check=True)
changes=[]
def replace(path, old, new):
    p=ROOT/path; s=p.read_text()
    if old not in s:
        raise RuntimeError(f'ไม่พบข้อความ patch ใน {path}; หยุดเพื่อไม่แก้โค้ดผิดตำแหน่ง')
    p.write_text(s.replace(old,new)); changes.append(path+': '+old[:80])
replace('utils/shape_predictor.py', 'PIL.Image.ANTIALIAS', 'PIL.Image.Resampling.LANCZOS')
replace('models/Alignment.py', 'torch.where(OB_region,', 'torch.where(OB_region.bool(),')
replace('models/Blending.py', 'loss.backward()', 'loss.sum().backward()')
replace('losses/masked_lpips/networks_basic.py', 'in_tens / torch.sum(mask)',
        'in_tens / torch.sum(mask).clamp_min(1e-8)')
# Extract the upstream native filter implementation, avoiding import-time CUDA compilation.
p=ROOT/'models/stylegan2/op/upfirdn2d.py'
s=p.read_text(); tree=ast.parse(s)
fn=next(x for x in tree.body if isinstance(x,ast.FunctionDef) and x.name=='upfirdn2d_native')
native=ast.get_source_segment(s, fn)
p.write_text('import torch\nfrom torch.nn import functional as F\n\n'+native+
    '\n\ndef upfirdn2d(input, kernel, up=1, down=1, pad=(0,0)):\n'
    '    return upfirdn2d_native(input, kernel, up, up, down, down, pad[0], pad[1], pad[0], pad[1])\n')
(ROOT/'models/stylegan2/op/fused_act.py').write_text(textwrap.dedent("""
import torch
from torch import nn
from torch.nn import functional as F
def fused_leaky_relu(input, bias, negative_slope=0.2, scale=2**0.5):
    if bias is not None:
        input = input + bias.view(1, -1, *([1]*(input.ndim-2)))
    return F.leaky_relu(input, negative_slope=negative_slope)*scale
class FusedLeakyReLU(nn.Module):
    def __init__(self, channel, negative_slope=0.2, scale=2**0.5):
        super().__init__()
        self.bias=nn.Parameter(torch.zeros(channel))
        self.negative_slope=negative_slope
        self.scale=scale
    def forward(self, input):
        return fused_leaky_relu(input,self.bias,self.negative_slope,self.scale)
"""))
replace('main.py', '    align = Alignment(args)',
        '    del ii2s\n    import gc\n    gc.collect(); torch.cuda.empty_cache()\n    align = Alignment(args)')
replace('main.py', '    blend = Blending(args)',
        '    del align\n    gc.collect(); torch.cuda.empty_cache()\n    blend = Blending(args)')
# Run trusted official checkpoints with legacy torch.load semantics in a separate process.
# Required because newer PyTorch defaults to weights_only=True.
(ROOT/'colab_run.py').write_text(textwrap.dedent("""
import os, random, runpy, numpy as np, torch
seed=int(os.environ.get('BARBERSHOP_SEED','3407'))
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark=False
torch.backends.cuda.matmul.allow_tf32=False
original_load=torch.load
def load_legacy(*args, **kwargs):
    kwargs.setdefault('weights_only',False)
    return original_load(*args, **kwargs)
torch.load=load_legacy
runpy.run_path('main.py',run_name='__main__')
"""))
changes += ['StyleGAN2 ops: upstream native filter + differentiable leaky ReLU',
            'colab_run.py: seeded run and legacy checkpoint load']
(ROOT/'colab_compatibility.json').write_text(json.dumps(changes,indent=2))
for path in patch_paths+['colab_run.py']:
    compile((ROOT/path).read_text(),path,'exec')
print('เตรียมโค้ดแล้ว:', COMMIT)

## 3. ดาวน์โหลด pretrained weights
โหลด StyleGAN2 FFHQ, BiSeNet และโมเดล landmark 68 จุด ส่วน PCA และ LPIPS weights อยู่ใน repository แล้ว VGG/ResNet ที่ใช้คำนวณ loss จะดาวน์โหลดเพิ่มเติมในการรันครั้งแรก

หาก Google Drive จำกัดโควตา เซลล์จะแจ้งชื่อไฟล์และเปิดให้อัปโหลดน้ำหนักที่ดาวน์โหลดจากลิงก์ทางการแทน ไม่ใช้ไฟล์ HTML ที่ดาวน์โหลดผิดเป็นโมเดล

In [ ]:
import gdown, bz2, shutil, requests
from google.colab import files
sys.path.insert(0,str(ROOT))
from utils.model_utils import weight_dic
weights=ROOT/'pretrained_models'; weights.mkdir(exist_ok=True)
def valid_weight(p):
    if not p.exists() or p.stat().st_size < 1_000_000: return False
    with p.open('rb') as f: head=f.read(256).lower()
    return b'<html' not in head and b'<!doctype' not in head
for name in ['ffhq.pt','seg.pth']:
    dest=weights/name
    if not valid_weight(dest):
        dest.unlink(missing_ok=True)
        try:
            gdown.download(weight_dic[name], str(dest), fuzzy=True)
        except Exception as e:
            print('ดาวน์โหลดไม่สำเร็จ:',e)
        if not valid_weight(dest):
            dest.unlink(missing_ok=True)
            print('ดาวน์โหลดไฟล์', name, 'จาก',weight_dic[name], 'แล้วเลือกไฟล์นี้เพื่ออัปโหลด')
            uploaded=files.upload()
            if len(uploaded)!=1: raise ValueError('เลือกน้ำหนักเพียงไฟล์เดียว')
            dest.write_bytes(next(iter(uploaded.values())))
    assert valid_weight(dest), 'ไฟล์น้ำหนักไม่สมบูรณ์: '+name
    print(name, round(dest.stat().st_size/1024**2,1),'MiB')
assert (weights/'ffhq_PCA.npz').exists(), 'ไม่พบ PCA จาก repository กรุณารันเซลล์ 2 ใหม่'
landmark=weights/'shape_predictor_68_face_landmarks.dat'
if not valid_weight(landmark):
    archive=weights/'landmarks.dat.bz2'
    url='https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2'
    with requests.get(url,stream=True,timeout=(30,300)) as r:
        r.raise_for_status()
        with archive.open('wb') as f:
            for block in r.iter_content(1024*1024): f.write(block)
    with bz2.open(archive,'rb') as src, landmark.open('wb') as dst:
        shutil.copyfileobj(src,dst)
    archive.unlink()
import dlib
predictor=dlib.shape_predictor(str(landmark))
print('โหลด landmark พร้อมใช้งาน')

## 4. อัปโหลดรูปทีละหน้าที่
แต่ละปุ่มให้เลือก **1 ภาพ** รองรับ JPG/PNG ภาพสีผมต้องเป็นภาพคนที่เห็นเส้นผมชัด สามารถเลือกภาพเดียวกับทรงผมได้หากต้องการทั้งทรงและสีจากต้นแบบเดียวกัน

In [ ]:
from PIL import Image, ImageOps
from IPython.display import display
from io import BytesIO
from datetime import datetime
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN=ROOT/'runs'/RUN_ID
RAW=RUN/'raw'; INPUT=RUN/'aligned'; OUT=RUN/'output'
for p in [RAW,INPUT,OUT]: p.mkdir(parents=True,exist_ok=True)
def upload_one(role,label):
    print(label)
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('กรุณาเลือกเพียง 1 ภาพ แล้วรันเซลล์นี้ใหม่')
    im=ImageOps.exif_transpose(Image.open(BytesIO(next(iter(uploaded.values()))))).convert('RGB')
    im.save(RAW/(role+'.png'))
    preview=im.copy(); preview.thumbnail((384,384)); display(preview)
for role,label in [('face','1/3 เลือกรูปหน้าของคุณ'),('shape','2/3 เลือกรูปทรงผม'),('color','3/3 เลือกรูปสีผม')]:
    upload_one(role,label)
print('Run:',RUN_ID)

## 5. จัดแนวใบหน้า (FFHQ alignment)
จัดตำแหน่งตาและปาก พร้อมครอปภาพเป็น 1024 × 1024 แบบที่โมเดลคาดหวัง หากตรวจไม่พบใบหน้าหรือพบหลายคนจะหยุด ให้กลับไปอัปโหลดรูปคนเดียว หน้าตรงขึ้น แล้วรันต่อใหม่ ตรวจดูว่าทรงผมไม่ถูกตัดมากเกินไป

In [ ]:
from utils.shape_predictor import align_face
import matplotlib.pyplot as plt
for role in ['face','shape','color']:
    aligned=align_face(str(RAW/(role+'.png')),predictor)
    if len(aligned)!=1:
        raise ValueError(f'{role}: พบ {len(aligned)} ใบหน้า กรุณาเปลี่ยนรูปในเซลล์ 4 ให้มีคนเดียวและหน้าชัด')
    aligned[0].convert('RGB').save(INPUT/(role+'.png'))
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,role in zip(axes,['face','shape','color']):
    ax.imshow(Image.open(INPUT/(role+'.png'))); ax.set_title(role.upper()); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. ตั้งค่าคุณภาพ
`official` ใช้จำนวนรอบตามค่าเริ่มต้นของโค้ด: W+ 1100, FS 250, alignment 140/100 และ blending 400
`preview` ลดรอบสำหรับทดสอบระบบเท่านั้น คุณภาพอาจลดลงอย่างชัดเจน และไม่ควรนำไปอ้างว่าเป็นผล baseline ทางการ

`realistic` เน้นความเป็นธรรมชาติ, `fidelity` เน้นสอดคล้องกับ mask เวลาไม่ตายตัว ขึ้นกับ GPU จำนวนรอบ และการดาวน์โหลดครั้งแรก อาจใช้เวลาหลายนาทีถึงหลักสิบนาทีหรือมากกว่า โดยเฉพาะเมื่อใช้ native ops

In [ ]:
QUALITY = 'official' # @param ['official', 'preview']
SIGN = 'realistic' # @param ['realistic', 'fidelity']
SEED = 3407 # @param {type:'integer'}
SMOOTH = 5 # @param {type:'integer'}
assert QUALITY in ['official','preview'] and SIGN in ['realistic','fidelity']
assert 1 <= SMOOTH <= 20
steps=({'W_steps':1100,'FS_steps':250,'align_steps1':140,'align_steps2':100,'blend_steps':400}
       if QUALITY=='official' else
       {'W_steps':200,'FS_steps':80,'align_steps1':60,'align_steps2':40,'blend_steps':100})
print(QUALITY,SIGN,steps)

## 7. สร้างภาพ
ลำดับภายใน: **W+ inversion → FS inversion → segmentation/semantic alignment → appearance blending → StyleGAN2 สร้างภาพ**

น้ำหนัก generator คงที่ การ optimization ที่เห็นคือการหา latent code ของภาพ ไม่ใช่ฝึก StyleGAN2 ใหม่ เซลล์แสดง progress และเก็บ log หากเกิดข้อผิดพลาดจะหยุด ไม่แสดงภาพค้างจากรอบก่อน

In [ ]:
import time, hashlib, platform
# A new output folder per attempt prevents stale results after a failed run.
ATTEMPT=OUT/datetime.now().strftime('%H%M%S_%f'); ATTEMPT.mkdir()
cmd=[sys.executable,'-u','colab_run.py','--input_dir',str(INPUT),'--output_dir',str(ATTEMPT),
     '--im_path1','face.png','--im_path2','shape.png','--im_path3','color.png',
     '--sign',SIGN,'--smooth',str(SMOOTH),'--seed',str(SEED)]
for k,v in steps.items(): cmd += ['--'+k,str(v)]
env=os.environ.copy(); env['BARBERSHOP_SEED']=str(SEED)
start=time.perf_counter()
with (ATTEMPT/'run.log').open('w') as log:
    process=subprocess.Popen(cmd,cwd=ROOT,env=env,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end=''); log.write(line); log.flush()
        status=process.wait()
    except BaseException:
        process.terminate(); process.wait(); raise
elapsed=time.perf_counter()-start
if status!=0:
    raise RuntimeError(f'สร้างภาพไม่สำเร็จ (exit {status}) อ่านข้อความท้าย log: {ATTEMPT / "run.log"}')
RESULT=ATTEMPT/f'face_shape_color_{SIGN}.png'
assert RESULT.exists(),'ไม่พบภาพผลลัพธ์ กรุณาตรวจ run.log'
with Image.open(RESULT) as im:
    im.verify()
metadata={'commit':COMMIT,'quality':QUALITY,'sign':SIGN,'seed':SEED,'smooth':SMOOTH,
          'steps':steps,'seconds_including_lazy_model_downloads':elapsed,
          'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
          'torchvision':torchvision.__version__,'python':platform.python_version(),
          'native_ops':True,'result':RESULT.name,
          'input_sha256':{r:hashlib.sha256((INPUT/(r+'.png')).read_bytes()).hexdigest()
                          for r in ['face','shape','color']}}
(ATTEMPT/'settings.json').write_text(json.dumps(metadata,indent=2))
print(f'เสร็จแล้ว: {elapsed/60:.1f} นาที | {Image.open(RESULT).size}')

## 8. ดูภาพและดาวน์โหลด PNG
ภาพผลลัพธ์และภาพเปรียบเทียบดาวน์โหลดแยกเป็นรูป ไม่บีบอัดเป็น ZIP เซลล์นี้ดาวน์โหลดผลลัพธ์ 1 รูปทันที ส่วนภาพเปรียบเทียบใช้เซลล์ถัดไป

In [ ]:
display(Image.open(RESULT))
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,label,p in zip(axes,['Face','Hair structure','Hair appearance','Result'],
                      [INPUT/'face.png',INPUT/'shape.png',INPUT/'color.png',RESULT]):
    ax.imshow(Image.open(p)); ax.set_title(label); ax.axis('off')
plt.tight_layout()
COMPARISON=ATTEMPT/'barbershop_comparison.png'
fig.savefig(COMPARISON,dpi=180,bbox_inches='tight',facecolor='white'); plt.show(); plt.close(fig)
DOWNLOAD=ATTEMPT/f'Barbershop_{QUALITY}_{SIGN}_{RUN_ID}.png'
shutil.copy2(RESULT,DOWNLOAD)
files.download(str(DOWNLOAD))

In [ ]:
# ดาวน์โหลดภาพเปรียบเทียบแยกอีก 1 รูป
files.download(str(COMPARISON))

## 9. เก็บลง Google Drive (เลือกทำ)
เปิด `SAVE_TO_DRIVE` หากต้องการเก็บผล ภาพเปรียบเทียบ และค่าการรันไว้ถาวร โดยไม่คัดลอกภาพต้นฉบับหรือ latent ทั้งหมด หากไม่ทำให้ดาวน์โหลดรูปก่อนปิด runtime เพราะไฟล์ใน /content จะหายเมื่อ session จบ

In [ ]:
SAVE_TO_DRIVE = False # @param {type:'boolean'}
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest=Path('/content/drive/MyDrive/Barbershop_results')/RUN_ID/ATTEMPT.name
    dest.mkdir(parents=True,exist_ok=True)
    for p in [DOWNLOAD,COMPARISON,ATTEMPT/'settings.json',ATTEMPT/'run.log',ROOT/'colab_compatibility.json']:
        shutil.copy2(p,dest/p.name)
    print('บันทึกแล้ว:',dest)
else:
    print('ข้ามการบันทึก Google Drive')

## สรุปให้เข้าใจง่าย
| ขั้นตอน | ทำอะไร |
|---|---|
| จัดแนวใบหน้า | วางตาและปากให้อยู่ในตำแหน่งที่โมเดลคุ้นเคย |
| W+ inversion | หาโค้ด 18 × 512 ที่ทำให้ StyleGAN2 วาดภาพคล้ายภาพอ้างอิง |
| FS inversion | เพิ่มแผนที่โครงสร้าง F ขนาด 32 × 32 × 512 เพื่อเก็บรายละเอียด และใช้ S คุม appearance |
| Segmentation | แยกตำแหน่งผม ใบหน้า และบริเวณอื่นด้วย BiSeNet |
| Semantic alignment | ปรับภาพใน latent space ให้บริเวณสำคัญตรงกับ mask เป้าหมาย และผสมโครงสร้าง |
| Blending | หาอัตราผสม appearance ให้ส่วนหน้าใกล้ภาพ face และส่วนผมใกล้ภาพ color ก่อนใช้โครงสร้างจาก shape |
| StyleGAN2 synthesis | วาดภาพรวม 1024 × 1024 จาก latent ที่ผสมแล้ว |

**สีผมไม่ได้ควบคุมด้วยรหัสสี:** ภาพ color ถ่ายทอด appearance รวมถึงโทนสีและลักษณะละเอียดของผม แสงและมุมหน้ามีผล จึงไม่รับประกันสีตรงทุกพิกเซล

**loss หลัก:** inversion ใช้ L2, LPIPS และ p-norm; alignment ใช้ segmentation cross-entropy และ masked style loss; blending ใช้ masked LPIPS สำหรับใบหน้าและเส้นผม ดูรายละเอียดจริงใน `losses/` ของ commit ที่ระบุ Notebook นี้คงขั้นตอนของ official code ไม่เพิ่ม diffusion หรือ upscaler

**งานวิจัยกับโค้ด:** อธิบายหลัก FS space และ semantic alignment จาก PDF v2 ที่แนบ ส่วนขั้นตอนที่รันอิง official repository ซึ่งมีการปรับปรุง semantic mask inpainting หลังเผยแพร่ PDF จึงไม่ควรถือว่าเป็นการทำซ้ำผลตัวเลขในบทความทุกประการ

### เมื่อพบปัญหา
- **CUDA ไม่พร้อม:** เลือก GPU แล้วรันจากเซลล์ 1 ใหม่
- **Torchvision import ไม่ได้:** เริ่ม Colab runtime ใหม่ ใช้ torch/torchvision คู่ที่ Colab ให้มา ไม่ติดตั้งทับแยกเวอร์ชัน
- **Google Drive quota/ดาวน์โหลดไม่สำเร็จ:** ใช้ลิงก์ที่เซลล์ 3 แสดงเพื่อดาวน์โหลดน้ำหนักเอง แล้วอัปโหลดตามคำแนะนำ
- **CUDA out of memory:** ลบ runtime แล้วเริ่มใหม่ ปิด notebook ที่ใช้ GPU อื่น หรือใช้ GPU หน่วยความจำมากขึ้น จำนวนรอบน้อยลงช่วยลดเวลาแต่ไม่รับประกันลดหน่วยความจำสูงสุด
- **ไม่เห็นสีหรือทรงตามต้นแบบ:** ตรวจภาพ aligned เลือกภาพที่มุมใกล้กันและเห็นเส้นผมชัด ใช้โหมด official หลีกเลี่ยงภาพผมถูกบังหรือผมเล็กมากจน mask หาย
- **ภาพไม่เหมือนใบหน้าเดิมทั้งหมด:** เป็นข้อจำกัดของ inversion และการผสม latent ไม่ใช่การแปะผมลงบนภาพเดิมโดยตรง
- **ต้องเปลี่ยนรูป:** กลับไปรันเซลล์ 4–8 ใหม่ แต่ละรอบมีโฟลเดอร์ใหม่ไม่ใช้ latent จากภาพก่อนหน้า

### แหล่งอ้างอิง
- Zhu et al. (2021), *Barbershop: GAN-based Image Compositing using Segmentation Masks*, arXiv:2106.01505v2 (PDF ที่ผู้ใช้แนบ), ส่วน 3 และรูป 2–4
- https://github.com/ZPdesu/Barbershop
- commit: `088d7d0f0623b348732fc15d8fbdea8b8cb730c7`
- https://github.com/ZPdesu/Barbershop/blob/088d7d0f0623b348732fc15d8fbdea8b8cb730c7/main.py
- `models/Embedding.py`, `models/Alignment.py`, `models/Blending.py`, `utils/model_utils.py`

โค้ดต้นฉบับอยู่ภายใต้ MIT License ของ repository การแก้เพื่อ Colab ระบุไว้ในเซลล์ 2 และบันทึกแยกไว้ในผลการรัน


# 10. ปรับภาพต่อด้วย Realistic Vision V5.1
รันส่วนนี้ **หลังได้ภาพ Barbershop แล้ว** หรือใส่ path/อัปโหลด PNG ผลลัพธ์เก่า ใช้ Stable Diffusion img2img เพื่อเติมรายละเอียดแบบเบา ไม่ต้องส่งรูป face/shape/color ใหม่

ค่าเริ่มต้น: **strength 0.15, guidance 3.5, 40 steps** (denoising จริงประมาณ 6 steps) และผสมกลับกับต้นฉบับ 70% diffusion + 30% original เพื่อลดการเปลี่ยนหน้า/สีผม ใช้ขนาด 1024 × 1024 เดิม ไม่มีขั้นลดเหลือ 512 แล้วขยายกลับ และไม่ได้เพิ่ม Real-ESRGAN

นี่เป็นการสร้างรายละเอียดใหม่ ไม่ใช่การกู้รายละเอียดจริงหรือรับประกันหน้าเหมือนเดิมทุกจุด Prompt อย่าง “same person” ไม่ใช่กลไกล็อก identity หากหน้า/ผมเปลี่ยน ให้ลด strength เป็น 0.10–0.12 หรือ REFINE_MIX เป็น 0.5 หากภาพแทบไม่เปลี่ยน ลอง strength 0.18–0.20 ทีละน้อย

Realistic Vision V5.1 ใช้ VAE แบบ ft-MSE ตามคำแนะนำผู้พัฒนา ส่วนเสริมนี้แยกจากวิธี Barbershop ในงานวิจัย ไม่ใช่ส่วนของ baseline เดิม **ยังไม่ได้ทดสอบ inference จริงบน GPU ในสภาพแวดล้อมผู้จัดทำ**

### 10.1 ติดตั้งแพ็กเกจเพิ่ม
ใช้ชุดเวอร์ชันที่ระบุ ไม่ติดตั้ง torch/torchvision ทับ ส่วน inference ทำใน process แยกจึงไม่ต้อง restart kernel และไม่เก็บโมเดล diffusion ค้างใน GPU เมื่อรันเสร็จ

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,'-m','pip','install','-q',
    'diffusers==0.35.1','transformers==4.56.2','accelerate==1.10.1',
    'safetensors>=0.4.3','huggingface_hub>=0.34.0,<1.0'],check=True)
print('ติดตั้งส่วนเสริมแล้ว')

### 10.2 เลือกภาพและปรับค่า
ปล่อย INPUT_IMAGE_PATH ว่างเพื่อใช้ `RESULT` จาก Barbershop อัตโนมัติ ถ้าไม่มี RESULT จะมีปุ่มอัปโหลดภาพ (ไม่ค้นหารูปเก่าแทนโดยเงียบ ๆ) รอบใหม่จะสร้างโฟลเดอร์ใหม่ทุกครั้ง

In [ ]:
from pathlib import Path
from datetime import datetime
from PIL import Image, ImageOps
from io import BytesIO
from google.colab import files
import json, math, hashlib
INPUT_IMAGE_PATH = '' # @param {type:'string'}
STRENGTH = 0.15 # @param {type:'number'}
GUIDANCE = 3.5 # @param {type:'number'}
STEPS = 40 # @param {type:'integer'}
REFINE_MIX = 0.70 # @param {type:'number'}
REFINE_SEED = 3407 # @param {type:'integer'}
PROMPT = 'realistic portrait photograph, natural skin texture, fine individual hair strands, detailed hair texture, natural lighting, subtle photographic detail'
NEGATIVE_PROMPT = 'blurry, out of focus, plastic skin, waxy skin, oversmoothed, cartoon, painting, deformed face, distorted eyes, unnatural hair, oversaturated, artifacts'
assert 0.05 <= STRENGTH <= 0.35, 'ใช้ strength 0.05 ถึง 0.35 เพื่อปรับอย่างระมัดระวัง'
assert 1 <= GUIDANCE <= 10 and 10 <= STEPS <= 100
assert 0 < REFINE_MIX <= 1 and int(STEPS * STRENGTH) >= 1
RV_DIR=Path('/content/Barbershop_refined')/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RV_DIR.mkdir(parents=True,exist_ok=True)
if INPUT_IMAGE_PATH.strip():
    source=Path(INPUT_IMAGE_PATH.strip())
    assert source.is_file(), 'ไม่พบ INPUT_IMAGE_PATH'
elif 'RESULT' in globals() and Path(RESULT).is_file():
    source=Path(RESULT)
else:
    print('เลือกรูปผลลัพธ์ Barbershop 1 ภาพ')
    up=files.upload()
    if len(up)!=1: raise ValueError('เลือกเพียง 1 ภาพ')
    source=RV_DIR/'uploaded.png'
    ImageOps.exif_transpose(Image.open(BytesIO(next(iter(up.values()))))).convert('RGB').save(source)
before=ImageOps.exif_transpose(Image.open(source)).convert('RGB')
if before.width % 8 or before.height % 8:
    raise ValueError('ขนาดภาพต้องหาร 8 ลงตัว แนะนำผล Barbershop 1024 × 1024 เพื่อไม่ต้อง resize')
if max(before.size)>1536:
    raise ValueError('ภาพใหญ่เกินค่าที่เตรียมไว้ กรุณาใช้ผล Barbershop 1024 × 1024')
BEFORE_PATH=RV_DIR/'Barbershop_before.png';before.save(BEFORE_PATH)
config={'input':str(BEFORE_PATH),'output_dir':str(RV_DIR),'model':'SG161222/Realistic_Vision_V5.1_noVAE',
        'vae':'stabilityai/sd-vae-ft-mse','strength':STRENGTH,'guidance':GUIDANCE,'steps':STEPS,
        'refine_mix':REFINE_MIX,'seed':REFINE_SEED,'prompt':PROMPT,'negative_prompt':NEGATIVE_PROMPT,
        'size':list(before.size),'input_sha256':hashlib.sha256(BEFORE_PATH.read_bytes()).hexdigest()}
(RV_DIR/'config.json').write_text(json.dumps(config,indent=2))
print('ภาพเข้า:',source,'ขนาด:',before.size,'| Denoising โดยประมาณ:',int(STEPS*STRENGTH))
print('โฟลเดอร์ผล:',RV_DIR)

### 10.3 สร้างภาพและจับเวลา
แยก **เวลาโหลดโมเดล** (รวมดาวน์โหลดในครั้งแรก), **เวลา diffusion** (รวม VAE encode/decode และ safety checker) และ **เวลาปรับภาพรวม** (รวมโหลดและบันทึก แต่ไม่รวมติดตั้งแพ็กเกจ/Barbershop)

เก็บทั้งภาพ diffusion โดยตรงและภาพผสมกับต้นฉบับ โดยภาพ “After” ที่ใช้แสดงคือภาพผสมตาม REFINE_MIX เปิด CPU offload และ VAE tiling ลด GPU memory การรันซ้ำยังโหลดโมเดลใหม่จาก cache ทุกครั้ง

In [ ]:
import textwrap, os, time
WORKER=RV_DIR/'refine_worker.py'
WORKER.write_text(textwrap.dedent("""
import json, sys, time, hashlib, importlib.metadata
from pathlib import Path
import torch
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from huggingface_hub import model_info
cfg=json.loads(Path(sys.argv[1]).read_text())
out=Path(cfg['output_dir'])
assert torch.cuda.is_available(), 'ต้องใช้ GPU runtime'
torch.backends.cudnn.benchmark=False
start=time.perf_counter()
# Resolve and record exact model revisions used for reproducibility.
model_rev=model_info(cfg['model']).sha
vae_rev=model_info(cfg['vae']).sha
vae=AutoencoderKL.from_pretrained(cfg['vae'],revision=vae_rev,torch_dtype=torch.float16,
                                use_safetensors=True)
pipe=StableDiffusionImg2ImgPipeline.from_pretrained(cfg['model'],revision=model_rev,
        vae=vae,torch_dtype=torch.float16,use_safetensors=True)
# Match VAE and pipeline dtypes for Stable Diffusion 1.5 img2img.
pipe.vae.to(dtype=torch.float16)
pipe.scheduler=DPMSolverMultistepScheduler.from_config(pipe.scheduler.config,
        algorithm_type='dpmsolver++',solver_order=2,use_karras_sigmas=True)
pipe.enable_model_cpu_offload()
pipe.enable_vae_tiling()
pipe.enable_vae_slicing()
torch.cuda.synchronize()
load_seconds=time.perf_counter()-start
before=Image.open(cfg['input']).convert('RGB')
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize(); tick=time.perf_counter()
generator=torch.Generator(device='cpu').manual_seed(cfg['seed'])
with torch.inference_mode():
    result=pipe(prompt=cfg['prompt'],negative_prompt=cfg['negative_prompt'],image=before,
        strength=cfg['strength'],num_inference_steps=cfg['steps'],guidance_scale=cfg['guidance'],
        generator=generator)
torch.cuda.synchronize(); inference_seconds=time.perf_counter()-tick
if result.nsfw_content_detected is not None and any(result.nsfw_content_detected):
    raise RuntimeError('Safety checker ไม่อนุญาตผลภาพนี้ จึงไม่บันทึกภาพดำเป็นผลสำเร็จ')
raw=result.images[0].convert('RGB')
if raw.size!=before.size: raise RuntimeError('ขนาดผลไม่ตรงต้นฉบับ: หยุดแทน resize โดยเงียบ')
raw.save(out/'RealisticVisionV51_raw.png')
after=Image.blend(before,raw,cfg['refine_mix'])
after.save(out/'Barbershop_RealisticVisionV51_after.png')
metrics={**cfg,'model_revision':model_rev,'vae_revision':vae_rev,
    'load_seconds':load_seconds,'diffusion_seconds':inference_seconds,
    'refinement_total_seconds':time.perf_counter()-start,
    'effective_denoising_steps':int(cfg['steps']*cfg['strength']),
    'peak_gpu_allocated_gib':torch.cuda.max_memory_allocated()/1024**3,
    'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
    'versions':{p:importlib.metadata.version(p) for p in ['diffusers','transformers','accelerate']}}
(out/'refinement_settings_and_time.json').write_text(json.dumps(metrics,indent=2))
print(json.dumps({k:metrics[k] for k in ['load_seconds','diffusion_seconds','refinement_total_seconds']},indent=2))
"""))
compile(WORKER.read_text(),str(WORKER),'exec')
# Clear any previous result variables: a failed attempt must never display an old image.
RV_AFTER_PATH=None
with (RV_DIR/'diffusion_run.log').open('w') as log:
    proc=subprocess.Popen([sys.executable,'-u',str(WORKER),str(RV_DIR/'config.json')],
                           stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in proc.stdout:
            print(line,end='');log.write(line);log.flush()
        rc=proc.wait()
    except BaseException:
        proc.terminate();proc.wait();raise
if rc!=0:
    raise RuntimeError(f'Diffusion ไม่สำเร็จ ดูข้อความท้าย log: {RV_DIR / "diffusion_run.log"}')
RV_AFTER_PATH=RV_DIR/'Barbershop_RealisticVisionV51_after.png'
assert RV_AFTER_PATH.is_file()
rv_metrics=json.loads((RV_DIR/'refinement_settings_and_time.json').read_text())
print('สร้างภาพปรับแล้วสำเร็จ')

### 10.4 เปรียบเทียบก่อน–หลัง พร้อมเวลา
ไม่มี ground truth สำหรับภาพทรงผมใหม่ จึงไม่ใช้ PSNR/SSIM มาอ้างว่าภาพ “ดีขึ้น” ให้ตรวจใบหน้า แนวผม สีผม และรายละเอียดที่อาจถูกสร้างเพิ่มด้วยตนเอง

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
assert RV_AFTER_PATH is not None and RV_AFTER_PATH.is_file(), 'รันเซลล์ 10.3 ให้สำเร็จก่อน'
rv_before=Image.open(BEFORE_PATH).convert('RGB');rv_after=Image.open(RV_AFTER_PATH).convert('RGB')
fig,axes=plt.subplots(1,2,figsize=(14,7))
axes[0].imshow(rv_before);axes[0].set_title('Before: Barbershop')
axes[1].imshow(rv_after);axes[1].set_title(f'After: Realistic Vision V5.1 | {rv_metrics["diffusion_seconds"]:.2f} s')
for ax in axes: ax.axis('off')
fig.suptitle(f'Strength {STRENGTH} | Mix {REFINE_MIX} | Same resolution {rv_before.width} x {rv_before.height}')
plt.tight_layout()
RV_COMPARE=RV_DIR/'Barbershop_RV51_before_after.png'
fig.savefig(RV_COMPARE,dpi=160,bbox_inches='tight',facecolor='white');plt.show();plt.close(fig)
print(f'โหลดโมเดล/ดาวน์โหลด: {rv_metrics["load_seconds"]:.2f} วินาที')
print(f'ปรับด้วย diffusion: {rv_metrics["diffusion_seconds"]:.2f} วินาที')
print(f'รวมโหลด + ปรับ + บันทึก: {rv_metrics["refinement_total_seconds"]:.2f} วินาที')
print('เวลาไม่รวมติดตั้งแพ็กเกจและสร้างภาพ Barbershop')
display(rv_after)

### 10.5 ดาวน์โหลดทีละภาพ และเลือกเก็บ Google Drive
เลือกไฟล์แล้วรันเซลล์ซ้ำเพื่อดาวน์โหลดแต่ละรูป ไม่บีบอัด ZIP ภาพหลังปรับเป็น PNG ขนาดเดิม

In [ ]:
DOWNLOAD_WHICH = 'after' # @param ['after', 'before', 'comparison', 'raw_diffusion', 'settings']
choices={'after':RV_AFTER_PATH,'before':BEFORE_PATH,'comparison':RV_COMPARE,
         'raw_diffusion':RV_DIR/'RealisticVisionV51_raw.png',
         'settings':RV_DIR/'refinement_settings_and_time.json'}
assert choices[DOWNLOAD_WHICH] is not None and choices[DOWNLOAD_WHICH].is_file()
files.download(str(choices[DOWNLOAD_WHICH]))

In [ ]:
SAVE_REFINED_TO_DRIVE = False # @param {type:'boolean'}
if SAVE_REFINED_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    target=Path('/content/drive/MyDrive/Barbershop_results/RealisticVisionV51')/RV_DIR.name
    target.mkdir(parents=True,exist_ok=True)
    for filename in ['Barbershop_before.png','RealisticVisionV51_raw.png',
                     'Barbershop_RealisticVisionV51_after.png','Barbershop_RV51_before_after.png',
                     'refinement_settings_and_time.json','diffusion_run.log']:
        shutil.copy2(RV_DIR/filename,target/filename)
    print('บันทึกแล้ว:',target)
else:
    print('ข้าม Google Drive; ดาวน์โหลดรูปก่อนปิด runtime')

### ข้อแนะนำและแหล่งอ้างอิงส่วนเสริม
- ถ้าต้องการรักษาหน้า/สีมากขึ้น: ลด strength หรือ REFINE_MIX ก่อน การเพิ่ม guidance ไม่ใช่การล็อกใบหน้า
- ถ้า GPU memory ไม่พอ: ใช้ GPU หน่วยความจำมากขึ้น ส่วนนี้เปิด CPU offload/tiling แล้ว ไม่มีการลดขนาดภาพอัตโนมัติ
- การรันครั้งแรกใช้เวลารวมดาวน์โหลดโมเดลหลาย GB ส่วนเวลา diffusion แยกไว้ ไม่ควรเทียบเวลารวมครั้งแรกกับการรันที่ cache พร้อมแล้ว
- ภาพ SD 1.5 ที่ 1024 อาจสร้างรายละเอียดแปลกได้ แม้ใช้ strength ต่ำ ให้ตรวจตา ฟัน แนวผม และปอยผมก่อนนำไปใช้
- หาก Hugging Face โหลดไม่สำเร็จ ให้ดู log และตรวจว่าเข้าถึง model repository ได้ อย่าเปลี่ยนไปใช้โมเดลอื่นแล้วรายงานว่าเป็น RV5.1
- การผสมกับต้นฉบับลดความเปลี่ยนแปลง แต่ไม่ใช่การรับประกัน identity; ค่า seed ช่วยทำซ้ำได้ภายใต้ environment เดิมและไม่รับประกันผลเหมือนกันข้าม GPU

แหล่งอ้างอิง: [Realistic Vision V5.1 model card](https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE), [ft-MSE VAE](https://huggingface.co/stabilityai/sd-vae-ft-mse), [Diffusers img2img](https://huggingface.co/docs/diffusers/using-diffusers/img2img)